In [1]:
import io
import boto3
import numpy as np
import pandas as pd
from urllib.parse import urlparse

S3_URI = "s3://smart-park-seattle/parking_v2/mvstgcn/year=2023/panels/week=2023-01-02/panel.npz"

def parse_s3(uri: str):
    u = urlparse(uri)
    return u.netloc, u.path.lstrip("/")

s3 = boto3.client("s3")
bucket, key = parse_s3(S3_URI)

raw = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
z = np.load(io.BytesIO(raw), allow_pickle=True)

print("NPZ keys:", list(z.keys()))
for k in z.files:
    v = z[k]
    if isinstance(v, np.ndarray):
        print(f"{k:20s} shape={v.shape} dtype={v.dtype}")
    else:
        print(f"{k:20s} type={type(v)}")

NPZ keys: ['X', 'y15', 'y30', 'mask', 'ts_utc', 'node_keys']
X                    shape=(672, 1512, 3) dtype=float32
y15                  shape=(672, 1512) dtype=int8
y30                  shape=(672, 1512) dtype=int8
mask                 shape=(672, 1512) dtype=uint8
ts_utc               shape=(672,) dtype=datetime64[ns]
node_keys            shape=(1512,) dtype=int64


In [2]:
# find a likely (T, N, F) tensor
X_key = None
for k in z.files:
    v = z[k]
    if isinstance(v, np.ndarray) and v.ndim == 3:
        X_key = k
        break

print("Selected X key:", X_key)
X = z[X_key]
print("X shape (T_in, N, F):", X.shape)

T, N, F = X.shape
print("T_in:", T, "N:", N, "F:", F)

# show last timestep: first 5 nodes, all features
print("\nX[last_t, first5_nodes, :]:")
print(X[T-1, :5, :])

# show time-series for node0 across time for first few features
print("\nX[:, node0, first 5 features]:")
print(X[:, 0, :min(5, F)])

Selected X key: X
X shape (T_in, N, F): (672, 1512, 3)
T_in: 672 N: 1512 F: 3

X[last_t, first5_nodes, :]:
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]

X[:, node0, first 5 features]:
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 ...
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]


In [3]:
cands = [k for k in z.files if "node" in k.lower() or "key" in k.lower() or "id" in k.lower()]
print("Candidate mapping keys:", cands)

for k in cands:
    v = z[k]
    if isinstance(v, np.ndarray):
        print(k, "shape:", v.shape, "dtype:", v.dtype)
        print("head:", v[:10])
        print("----")

Candidate mapping keys: ['node_keys']
node_keys shape: (1512,) dtype: int64
head: [1001 1002 1005 1006 1009 1013 1014 1017 1018 1021]
----
